# Lotin va kirill yozuvi: o'zbek NLP modellarining chidamliligi

SIB-200 (kodlovchilar) va Belebele (LLM) bo'yicha lotin / lotin-normallashgan / kirill (qoidaga asoslangan o'girish) shartlari.
Notebook ketma-ket ishlaydi (Save & Run All mumkin). `SMOKE = True` tezkor sinov, `False` to'liq tajriba.
Natijalar `/kaggle/working/results` va `results.zip` ga yoziladi.

In [ ]:
import os, sys, subprocess, json, glob, shutil, time
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else os.getcwd()
os.chdir(WORK); sys.path.insert(0, WORK)

SMOKE = False           # True: tezkor sinov (2 epoxa, 1 seed, 20 ta LLM savoli); to'liq tajribada False

ENCODER_MODELS = [
    "FacebookAI/xlm-roberta-base",
    "google-bert/bert-base-multilingual-cased",
    "tahrirchi/tahrirchi-bert-base",
    "tahrirchi/tahrirchi-bert-small",
]
LLM_MODELS = ["Qwen/Qwen3-8B", "Qwen/Qwen2.5-7B-Instruct"]   # alloma tokenizatori kirillni tashlaydi: asosiy taqqoslashdan chiqarilgan
LLM_SHOTS = [0] if SMOKE else [0, 5]
LLM_4BIT = True
RUN_ENCODERS = True
RUN_DIAGNOSTICS = True
RUN_LLM = True

SEEDS = [13] if SMOKE else [13, 21, 42, 7, 99]
EPOCHS = 2 if SMOKE else 10
LRS = ["3e-5", "1e-4"] if SMOKE else ["3e-5", "5e-5", "1e-4"]
LLM_LIMIT = 20 if SMOKE else None

RUN_LLM = False
LRS = ["1e-5", "3e-5", "5e-5", "1e-4"]


def run(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if any(w in line for w in ("Warning", "warn(", "FutureWarning")):
            continue
        print(line, end="")
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"command failed with code {p.returncode}: {cmd}")

import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("SMOKE =", SMOKE, "| seeds =", SEEDS, "| epochs =", EPOCHS, "| encoders =", len(ENCODER_MODELS), "| LLMs =", LLM_MODELS)

In [ ]:
run("pip install -q accelerate bitsandbytes scipy scikit-learn pytest datasets")
import transformers, datasets
print("transformers", transformers.__version__, "| datasets", datasets.__version__)

## Skriptlar (%%writefile). Ularni o'zgartirish shart emas.

In [ ]:
%%writefile translit_uz.py
"""
translit_uz.py -- rule-based Uzbek Latin <-> Cyrillic transliteration.

Design goals
------------
* Deterministic and fully documented (so the paper can state the exact rules).
* Latin -> Cyrillic is the direction used in the experiments.
* Cyrillic -> Latin is provided for round-trip diagnostics only.

KNOWN, UNAVOIDABLE LIMITATIONS (must be reported in the paper)
--------------------------------------------------------------
1. Russian loanwords: Cyrillic Uzbek keeps etymological letters (ц, щ, ь, ы, ъ in
   some loans, ё, ю, я in loans) that the Latin script has already merged
   (e.g. s for both с and ц). A Latin->Cyrillic mapping therefore CANNOT recover them.
   A small stem list (TS_STEMS) restores ц in the commonest -tsiya/-tsion words;
   everything else is rendered with the regular mapping.
2. 'sh', 'ch', 'ng' digraphs are treated as single units; the rare sequences s+h,
   c+h, n+g (e.g. 'mashhur' is fine, 'ongli' is fine) are not disambiguated.
3. 'ye/yo/yu/ya' -> е/ё/ю/я, 'y' + other -> й.
4. 'e' -> э at word start or after a vowel, otherwise е.
5. Apostrophe-like characters are normalised first (see APOSTROPHES):
   after o/g -> modifier (ў/ғ); elsewhere -> tutuq belgisi (ъ).
   NOTE: this normalisation is itself a typographic intervention; the paper's
   "latin_norm" control condition isolates it from the script change.
"""
from __future__ import annotations

import re
import unicodedata

# characters that all stand for the Uzbek modifier / tutuq in real-world text
APOSTROPHES = "'`´ʻʼ‘’ʹ′ʻʼ‘’"
_APOS_RE = re.compile("[" + re.escape(APOSTROPHES) + "]")
CANON_APOS = "ʻ"  # we normalise to U+02BB internally

LAT_VOWELS = set("aeiouAEIOU")
CYR_VOWELS = set("аеёиоуэюяўАЕЁИОУЭЮЯЎ")

# Russian-origin stems where Latin 'ts' corresponds to Cyrillic 'ц'
TS_STEMS = [
    "tsiya", "tsiy", "tsion", "tsent", "tsikl", "tsirk", "tsement", "tsiv",
    "militsiy", "polytsi", "sentsion",
]

# ---------------------------------------------------------------- Latin -> Cyrillic
_L2C_SINGLE = {
    "a": "а", "b": "б", "d": "д", "f": "ф", "g": "г", "h": "ҳ", "i": "и",
    "j": "ж", "k": "к", "l": "л", "m": "м", "n": "н", "o": "о", "p": "п",
    "q": "қ", "r": "р", "s": "с", "t": "т", "u": "у", "v": "в", "x": "х",
    "y": "й", "z": "з",
    # letters occasionally present in loanwords / proper names
    "c": "к", "w": "в",
}


def normalize_apostrophes(text: str, to: str = CANON_APOS) -> str:
    """Map every apostrophe-like character to a single canonical character."""
    return _APOS_RE.sub(to, text)


def _case_like(src: str, out: str) -> str:
    return out.upper() if src[:1].isupper() else out


def _safe_lower(t: str) -> str:
    """Per-character lowercase that never changes the string length
    (str.lower() can expand e.g. 'İ' to two characters and break index alignment)."""
    return "".join(c.lower() if len(c.lower()) == 1 else c for c in t)


def lat2cyr(text: str) -> str:
    """Latin Uzbek -> Cyrillic Uzbek (see module docstring for limitations)."""
    t = normalize_apostrophes(unicodedata.normalize("NFC", text))
    out: list[str] = []
    n = len(t)
    i = 0
    prev_is_vowel = False  # previous *Latin* char was a vowel letter
    at_word_start = True
    low = _safe_lower(t)

    while i < n:
        ch = t[i]
        lc = low[i]
        nxt = low[i + 1] if i + 1 < n else ""
        nxt2 = low[i + 2] if i + 2 < n else ""

        if not ch.isalpha() and ch != CANON_APOS:
            out.append(ch)
            at_word_start = True
            prev_is_vowel = False
            i += 1
            continue

        # ---- ts-stems (Russian loans)
        if lc == "t" and nxt == "s":
            rest = low[i:i + 8]
            if any(rest.startswith(stem) for stem in TS_STEMS):
                out.append(_case_like(ch, "ц"))
                i += 2
                at_word_start = False
                prev_is_vowel = False
                continue

        # ---- modifier letters oʻ, gʻ
        if lc in "og" and nxt == CANON_APOS:
            out.append(_case_like(ch, "ў" if lc == "o" else "ғ"))
            i += 2
            at_word_start = False
            prev_is_vowel = (lc == "o")
            continue

        # ---- digraphs sh, ch, ng
        if lc == "s" and nxt == "h":
            out.append(_case_like(ch, "ш")); i += 2
            at_word_start = prev_is_vowel = False; continue
        if lc == "c" and nxt == "h":
            out.append(_case_like(ch, "ч")); i += 2
            at_word_start = prev_is_vowel = False; continue
        if lc == "n" and nxt == "g":
            # 'ng' -> 'нг' (two Cyrillic letters)
            if ch.isupper() and t[i + 1].isupper():
                out.append("НГ")
            elif ch.isupper():
                out.append("Нг")
            else:
                out.append("нг")
            i += 2
            at_word_start = prev_is_vowel = False; continue

        # ---- y + vowel
        if lc == "y":
            if nxt == "e":
                out.append(_case_like(ch, "е")); i += 2
                at_word_start = False; prev_is_vowel = True; continue
            if nxt == "o" and nxt2 != CANON_APOS:
                out.append(_case_like(ch, "ё")); i += 2
                at_word_start = False; prev_is_vowel = True; continue
            if nxt == "u":
                out.append(_case_like(ch, "ю")); i += 2
                at_word_start = False; prev_is_vowel = True; continue
            if nxt == "a":
                out.append(_case_like(ch, "я")); i += 2
                at_word_start = False; prev_is_vowel = True; continue
            out.append(_case_like(ch, "й")); i += 1
            at_word_start = False; prev_is_vowel = False; continue

        # ---- e
        if lc == "e":
            out.append(_case_like(ch, "э" if (at_word_start or prev_is_vowel) else "е"))
            i += 1
            at_word_start = False; prev_is_vowel = True; continue

        # ---- standalone apostrophe -> tutuq belgisi
        if ch == CANON_APOS:
            out.append("ъ"); i += 1
            at_word_start = False; prev_is_vowel = False; continue

        # ---- plain letters
        if lc in _L2C_SINGLE:
            out.append(_case_like(ch, _L2C_SINGLE[lc]))
            prev_is_vowel = lc in "aiou"
        else:
            out.append(ch)  # already Cyrillic or other script: keep
            prev_is_vowel = ch in CYR_VOWELS
        at_word_start = False
        i += 1

    return "".join(out)


# ---------------------------------------------------------------- Cyrillic -> Latin
_C2L_SINGLE = {
    "а": "a", "б": "b", "в": "v", "г": "g", "д": "d", "ж": "j", "з": "z",
    "и": "i", "й": "y", "к": "k", "л": "l", "м": "m", "н": "n", "о": "o",
    "п": "p", "р": "r", "с": "s", "т": "t", "у": "u", "ф": "f", "х": "x",
    "ч": "ch", "ш": "sh", "э": "e", "ю": "yu", "я": "ya", "ё": "yo",
    "ў": "o" + CANON_APOS, "қ": "q", "ғ": "g" + CANON_APOS, "ҳ": "h",
    "ъ": CANON_APOS, "ь": "", "ы": "i", "ц": "s", "щ": "shch",
}


def cyr2lat(text: str) -> str:
    out: list[str] = []
    prev_cyr_vowel = False
    at_word_start = True
    for ch in text:
        lc = ch.lower()
        if lc == "е":  # Cyrillic Ie
            r = "ye" if (at_word_start or prev_cyr_vowel) else "e"
            out.append(r.capitalize() if ch.isupper() else r)
        elif lc in _C2L_SINGLE:
            r = _C2L_SINGLE[lc]
            out.append(r.capitalize() if (ch.isupper() and r) else r)
        else:
            out.append(ch)
        if ch.isalpha():
            at_word_start = False
            prev_cyr_vowel = lc in CYR_VOWELS
        else:
            at_word_start = True
            prev_cyr_vowel = False
    return "".join(out)


# ---------------------------------------------------------------- helpers for the experiments
def latin_normalised(text: str) -> str:
    """CONTROL condition: Latin text with apostrophes canonicalised to ASCII "'".
    Isolates the effect of typographic normalisation from the script change."""
    return normalize_apostrophes(text, to="'")


def script_share(text: str) -> dict:
    """Return fraction of alphabetic characters per script (Latin / Cyrillic / other)."""
    lat = cyr = oth = 0
    for ch in text:
        if not ch.isalpha():
            continue
        name = unicodedata.name(ch, "")
        if name.startswith("LATIN"):
            lat += 1
        elif name.startswith("CYRILLIC"):
            cyr += 1
        else:
            oth += 1
    tot = max(lat + cyr + oth, 1)
    return {"latin": lat / tot, "cyrillic": cyr / tot, "other": oth / tot}


def char_error_rate(ref: str, hyp: str) -> float:
    """Levenshtein CER (no external deps)."""
    if not ref:
        return 0.0 if not hyp else 1.0
    prev = list(range(len(hyp) + 1))
    for i, rc in enumerate(ref, 1):
        cur = [i]
        for j, hc in enumerate(hyp, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (rc != hc)))
        prev = cur
    return prev[-1] / len(ref)


if __name__ == "__main__":
    demo = "Oʻzbekiston Respublikasi, gʻarbda Yevropa; yoʻq, ma'no, ekran, ongli, Toshkent, informatsiya."
    print(demo)
    print(lat2cyr(demo))
    print(cyr2lat(lat2cyr(demo)))

In [ ]:
%%writefile test_translit.py
"""Unit tests for translit_uz.  Run:  python -m pytest -q test_translit.py"""
import pytest
from translit_uz import lat2cyr, cyr2lat, latin_normalised, script_share

CASES = [
    ("Oʻzbekiston", "Ўзбекистон"),
    ("o'zbek", "ўзбек"),
    ("gʻarb", "ғарб"),
    ("yoʻq", "йўқ"),
    ("yo'l", "йўл"),
    ("yosh", "ёш"),
    ("yangi", "янги"),
    ("yuz", "юз"),
    ("yer", "ер"),
    ("ma'no", "маъно"),
    ("shahar", "шаҳар"),
    ("choy", "чой"),
    ("ongli", "онгли"),
    ("ekran", "экран"),
    ("teatr", "театр"),
    ("poezd", "поэзд"),
    ("Toshkent", "Тошкент"),
    ("qalam", "қалам"),
    ("xalq", "халқ"),
    ("hayot", "ҳаёт"),
    ("informatsiya", "информация"),
    ("ketsa", "кетса"),         # 'ts' must NOT become ц here
    ("Rossiya", "Россия"),
    ("SHAHAR", "ШАҲАР"),
    ("Shahar", "Шаҳар"),
    ("1-sentabr, 2024!", "1-сентабр, 2024!"),
]


@pytest.mark.parametrize("lat,cyr", CASES)
def test_lat2cyr(lat, cyr):
    assert lat2cyr(lat) == cyr


def test_apostrophe_variants_equal():
    variants = ["o'zbek", "oʻzbek", "o’zbek", "o‘zbek", "o`zbek", "oʼzbek"]
    assert len({lat2cyr(v) for v in variants}) == 1


def test_no_latin_left():
    s = lat2cyr("Turkiya uch tomondan dengizlar bilan oʻralgan: gʻarbda Egey dengizi.")
    assert script_share(s)["latin"] == 0.0


def test_roundtrip_mostly_identity():
    for lat in ["oʻzbek tili", "gʻoya", "yoʻl", "yangi kun", "shahar", "maʼno"]:
        back = cyr2lat(lat2cyr(lat))
        assert latin_normalised(back).replace("ʻ", "'").replace("ʼ", "'") == \
               latin_normalised(lat).replace("ʻ", "'").replace("ʼ", "'")

In [ ]:
%%writefile prepare_data.py
"""
prepare_data.py -- build the 3 script conditions for SIB-200 (uzn_Latn) and Belebele (uzn_Latn).

Conditions
----------
latn       : original text, untouched
latn_norm  : CONTROL -- same Latin text, apostrophe-like characters canonicalised to "'"
             (isolates typographic normalisation from the script change)
cyrl       : rule-based Latin->Cyrillic transliteration (translit_uz.lat2cyr)

Usage
-----
python prepare_data.py --out data            # SIB-200 only (GitHub raw files, no HF needed)
python prepare_data.py --out data --belebele # also Belebele (needs `datasets` + internet)

Writes:  data/sib200_{split}_{condition}.tsv  (index_id, category, text)
         data/belebele_{condition}.jsonl
         data/prep_report.json
"""
import argparse, json, os, urllib.request
from collections import Counter

from translit_uz import lat2cyr, latin_normalised, script_share, normalize_apostrophes

SIB_URL = "https://raw.githubusercontent.com/dadelani/sib-200/main/data/annotated/uzn_Latn/{split}.tsv"
SPLIT_FILES = {"train": "train", "validation": "dev", "test": "test"}


def fetch(url: str) -> str:
    with urllib.request.urlopen(url, timeout=60) as r:
        return r.read().decode("utf-8")


def transform_all(text: str) -> dict:
    return {"latn": text, "latn_norm": latin_normalised(text), "cyrl": lat2cyr(text)}


def prep_sib(out: str, report: dict):
    rep = {}
    for split, fname in SPLIT_FILES.items():
        raw = fetch(SIB_URL.format(split=fname))
        # NOTE: parse by hand. 78 SIB-200 lines contain a double quote, which the csv module
        # (default quoting) would treat as a quote character and corrupt.
        lines = [l for l in raw.replace("\r\n", "\n").split("\n") if l.strip()]
        header = lines[0].split("\t")
        assert header == ["index_id", "category", "text"], header
        rows = []
        for l in lines[1:]:
            parts = l.split("\t")
            assert len(parts) == 3, f"bad line: {l[:80]!r}"
            rows.append(dict(zip(header, parts)))
        for cond in ["latn", "latn_norm", "cyrl"]:
            path = os.path.join(out, f"sib200_{split}_{cond}.tsv")
            with open(path, "w", encoding="utf-8", newline="") as f:
                f.write("\t".join(header) + "\n")
                for r in rows:
                    t = transform_all(r["text"])[cond]
                    assert "\t" not in t and "\n" not in t
                    f.write(f"{r['index_id']}\t{r['category']}\t{t}\n")
        # sanity statistics on this split
        cyr_text = " ".join(lat2cyr(r["text"]) for r in rows)
        rep[split] = {
            "n": len(rows),
            "labels": dict(Counter(r["category"] for r in rows)),
            "cyrl_script_share": script_share(cyr_text),
            "latn_script_share": script_share(" ".join(r["text"] for r in rows)),
        }
    report["sib200"] = rep


def prep_belebele(out: str, report: dict):
    from datasets import load_dataset
    ds = load_dataset("facebook/belebele", "uzn_Latn", split="test")
    fields_to_convert = ["flores_passage", "question", "mc_answer1", "mc_answer2",
                         "mc_answer3", "mc_answer4"]
    need = fields_to_convert + ["correct_answer_num"]
    missing = [c for c in need if c not in ds.column_names]
    assert not missing, f"Belebele columns changed, missing {missing}; found {ds.column_names}"
    assert len(ds) == 900, f"expected 900 Belebele items, got {len(ds)}"
    # `correct_answer_num` may be a ClassLabel (HF returns 0..3) or a plain string ("1".."4").
    # Normalise to the string "1".."4" that exp_llm_belebele.py expects.
    feat = ds.features["correct_answer_num"]

    def gold_str(v):
        if hasattr(feat, "int2str") and isinstance(v, int):
            v = feat.int2str(v)
        v = str(v)
        assert v in {"1", "2", "3", "4"}, f"unexpected correct_answer_num value: {v!r}"
        return v

    keep_meta = [c for c in ("link", "question_number") if c in ds.column_names]
    for cond in ["latn", "latn_norm", "cyrl"]:
        path = os.path.join(out, f"belebele_{cond}.jsonl")
        with open(path, "w", encoding="utf-8") as f:
            for ex in ds:
                # only JSON-safe fields (the `ds` timestamp column would break json.dumps)
                d = {k: ex[k] for k in keep_meta}
                for k in fields_to_convert:
                    d[k] = transform_all(ex[k])[cond]
                d["correct_answer_num"] = gold_str(ex["correct_answer_num"])
                f.write(json.dumps(d, ensure_ascii=False) + "\n")
    report["belebele"] = {"n": len(ds), "columns": ds.column_names,
                          "correct_answer_feature": type(feat).__name__}


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="data")
    ap.add_argument("--belebele", action="store_true")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    report = {}
    prep_sib(a.out, report)
    if a.belebele:
        prep_belebele(a.out, report)
    with open(os.path.join(a.out, "prep_report.json"), "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    print(json.dumps(report, ensure_ascii=False, indent=2))

In [ ]:
%%writefile exp_encoder_sib200.py
"""
exp_encoder_sib200.py -- script-robustness of encoder models on SIB-200 (uzn), Kaggle-ready.

Design (train script x test script matrix)
------------------------------------------
train condition : latn | cyrl | mixed (latn + cyrl copies of the same training sentences)
test conditions : latn | latn_norm (control) | cyrl   (all evaluated for every trained model)

Protocol (kept identical to the author's earlier news-classification protocol)
------------------------------------------------------------------------------
* Same lr grid for every model: 3e-5, 5e-5, 1e-4
* Pilot : seed 13, lr selected SEPARATELY per training condition (latn, cyrl, mixed) on validation
          Macro-F1 of that condition (removes the lr confound of the first run)
* Final : each training condition uses its own selected lr
* Checkpoint selection per run: best validation Macro-F1 *in the training condition*
* Final seeds: 13 21 42 7 99  (n=5; SIB-200 test has only 204 items, so more seeds help)

Usage (Kaggle cell)
-------------------
!python exp_encoder_sib200.py --model xlm-roberta-base --stage all
!python exp_encoder_sib200.py --model bert-base-multilingual-cased --stage all
# Tahrirchi model ids: set exactly as on Hugging Face  [VERIFICATION REQUIRED]
!python exp_encoder_sib200.py --model <tahrirchi-model-id> --stage all

Outputs (per model): results/encoder/<slug>/{pilot.json, runs.jsonl}
runs.jsonl keeps item-level predictions so that paired tests/bootstraps can be computed later.
"""
import argparse, copy, json, os, random, time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          get_linear_schedule_with_warmup)

TEST_CONDS = ["latn", "latn_norm", "cyrl"]
DEV = "cuda" if torch.cuda.is_available() else "cpu"


# ----------------------------------------------------------------------------- data
def load_split(data_dir, split, cond):
    return pd.read_csv(os.path.join(data_dir, f"sib200_{split}_{cond}.tsv"), sep="\t",
                       quoting=3, dtype={"index_id": int})  # quoting=3: QUOTE_NONE (texts contain quotes)


def get_xy(data_dir, split, cond, label2id):
    if cond == "mixed":
        df = pd.concat([load_split(data_dir, split, "latn"), load_split(data_dir, split, "cyrl")],
                       ignore_index=True)
    else:
        df = load_split(data_dir, split, cond)
    return df["text"].astype(str).tolist(), [label2id[c] for c in df["category"]]


def make_loader(tok, texts, labels, max_len, bs, shuffle):
    enc = tok(texts, truncation=True, max_length=max_len)
    items = [{**{k: enc[k][i] for k in enc}, "labels": labels[i]} for i in range(len(texts))]
    n_trunc = sum(len(x) >= max_len for x in enc["input_ids"])

    def collate(batch):
        # IMPORTANT: copy; popping from the shared dict would remove 'labels' for epoch 2+
        batch = [dict(b) for b in batch]
        lab = torch.tensor([b.pop("labels") for b in batch])
        out = tok.pad(batch, return_tensors="pt")
        out["labels"] = lab
        return out
    dl = DataLoader(items, batch_size=bs, shuffle=shuffle, collate_fn=collate)
    dl.n_truncated = n_trunc  # sentences that hit max_len (must be compared across scripts)
    return dl


# ----------------------------------------------------------------------------- train / eval
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds = []
    for b in loader:
        b = {k: v.to(DEV) for k, v in b.items() if k != "labels"}
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEV == "cuda")):
            logits = model(**b).logits
        preds += logits.argmax(-1).cpu().tolist()
    return preds


def run_one(args, tok, label2id, train_cond, lr, seed, test_conds):
    set_seed(seed)
    n_lab = len(label2id)
    model = AutoModelForSequenceClassification.from_pretrained(
        args.model, num_labels=n_lab, trust_remote_code=args.trust_remote_code).to(DEV)
    xtr, ytr = get_xy(args.data_dir, "train", train_cond, label2id)
    xva, yva = get_xy(args.data_dir, "validation", train_cond, label2id)
    tr = make_loader(tok, xtr, ytr, args.max_len, args.bs, True)
    va = make_loader(tok, xva, yva, args.max_len, 64, False)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total = len(tr) * args.epochs
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEV == "cuda"))

    best_f1, best_state, best_ep = -1.0, None, -1
    for ep in range(args.epochs):
        model.train()
        for b in tr:
            b = {k: v.to(DEV) for k, v in b.items()}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEV == "cuda")):
                loss = model(**b).loss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
        f1 = f1_score(yva, predict(model, va), average="macro",
                      labels=list(range(n_lab)), zero_division=0)
        if f1 > best_f1:
            best_f1, best_ep = f1, ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)

    out = {"val_macro_f1": best_f1, "best_epoch": best_ep + 1,
           # training collapse (model predicts ~one class): best validation Macro-F1 < 0.2.
           # Flagged, NOT retried, so the fairness protocol stays unchanged; report it.
           "collapsed": bool(best_f1 < 0.2), "tests": {}}
    for tc in test_conds:
        xte, yte = get_xy(args.data_dir, "test", tc, label2id)
        te = make_loader(tok, xte, yte, args.max_len, 64, False)
        p = predict(model, te)
        out["tests"][tc] = {
            "acc": accuracy_score(yte, p),
            "macro_f1": f1_score(yte, p, average="macro", labels=list(range(n_lab)), zero_division=0),
            "n_classes_never_predicted": int(n_lab - len(set(p))),
            "n_truncated": te.n_truncated,
            "preds": p, "labels": yte,
        }
    del model, opt
    torch.cuda.empty_cache()
    return out


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--data_dir", default="data")
    ap.add_argument("--out_dir", default="results/encoder")
    ap.add_argument("--stage", choices=["pilot", "final", "all"], default="all")
    ap.add_argument("--lrs", type=float, nargs="+", default=[3e-5, 5e-5, 1e-4])
    ap.add_argument("--pilot_seed", type=int, default=13)
    ap.add_argument("--seeds", type=int, nargs="+", default=[13, 21, 42, 7, 99])
    ap.add_argument("--train_conds", nargs="+", default=["latn", "cyrl", "mixed"])
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--bs", type=int, default=16)
    # 256, not 128: Cyrillic text yields more subword tokens for some tokenizers, so a short
    # limit would truncate Cyrillic more than Latin (a hidden confound). n_truncated is logged.
    ap.add_argument("--max_len", type=int, default=256)
    ap.add_argument("--trust_remote_code", action="store_true")
    args = ap.parse_args()

    slug = args.model.replace("/", "__")
    odir = os.path.join(args.out_dir, slug)
    os.makedirs(odir, exist_ok=True)
    tok = AutoTokenizer.from_pretrained(args.model, trust_remote_code=args.trust_remote_code)
    labels = sorted(load_split(args.data_dir, "train", "latn")["category"].unique())
    label2id = {c: i for i, c in enumerate(labels)}
    pilot_path = os.path.join(odir, "pilot.json")

    # ---- pilot: lr selection SEPARATELY for each training condition (val in that condition, one seed)
    if args.stage in ("pilot", "all"):
        sel = json.load(open(pilot_path)) if os.path.exists(pilot_path) else {}
        grid_all = sel.get("grid_by_train", {})
        for tcond in args.train_conds:
            res = grid_all.get(tcond, {})
            for lr in args.lrs:
                if repr(lr) in res:
                    continue  # resume
                r = run_one(args, tok, label2id, tcond, lr, args.pilot_seed, [])
                res[repr(lr)] = r["val_macro_f1"]
                print(f"[pilot] {args.model} train={tcond} lr={lr} val_macroF1={r['val_macro_f1']:.4f}", flush=True)
            grid_all[tcond] = res
            json.dump({"grid_by_train": grid_all, "pilot_seed": args.pilot_seed,
                       "selected_lr_by_train": {t: float(max(g, key=g.get)) for t, g in grid_all.items()},
                       "note": "per-train-condition lr, selected on validation Macro-F1 in that condition; "
                               "ties -> first (lowest) lr in grid order"},
                      open(pilot_path, "w"), indent=2)

    # ---- final: each training condition uses its own selected lr
    if args.stage in ("final", "all"):
        if not os.path.exists(pilot_path):
            raise SystemExit(f"{pilot_path} missing: run --stage pilot (or all) first")
        lr_by = json.load(open(pilot_path))["selected_lr_by_train"]
        runs_path = os.path.join(odir, "runs.jsonl")
        done = set()  # resume support: never duplicate a (train_cond, seed, lr) run
        if os.path.exists(runs_path):
            for line in open(runs_path, encoding="utf-8"):
                d = json.loads(line)
                done.add((d["train_cond"], d["seed"], d["lr"]))
        with open(runs_path, "a", encoding="utf-8") as f:
            for tcond in args.train_conds:
                lr = lr_by[tcond]
                for seed in args.seeds:
                    if (tcond, seed, lr) in done:
                        print(f"[skip] {tcond} seed={seed} already done", flush=True)
                        continue
                    t0 = time.time()
                    r = run_one(args, tok, label2id, tcond, lr, seed, TEST_CONDS)
                    rec = {"model": args.model, "train_cond": tcond, "lr": lr, "seed": seed,
                           "label_names": labels, **r, "seconds": round(time.time() - t0, 1)}
                    f.write(json.dumps(rec) + "\n"); f.flush()
                    print(f"[final] {args.model} train={tcond} lr={lr} seed={seed} " +
                          " ".join(f"{k}:{v['macro_f1']:.3f}" for k, v in r["tests"].items()), flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile analyze_encoder.py
"""
analyze_encoder.py -- aggregate encoder runs into paper tables.

Reads results/encoder/*/runs.jsonl and writes results/tables/*.csv

Per model x train_cond x test_cond:  mean +- std Accuracy and Macro-F1 over seeds
Delta (script effect) per model/train_cond:
    D_script  = F1(test=cyrl)      - F1(test=latn)        # total effect
    D_typo    = F1(test=latn_norm) - F1(test=latn)        # typographic control
    D_pure    = F1(test=cyrl)      - F1(test=latn_norm)   # script effect net of apostrophe normalisation
Statistics (reported with their limitations):
  * paired t-test over seeds (n = number of seeds; low power)
  * item-level paired bootstrap CI of Delta (10,000 resamples of the 204 test items,
    applied to each seed and averaged)
  * Holm correction across models within each (train_cond, delta)
"""
import glob, json, os, sys
import numpy as np
import pandas as pd
from scipy import stats

IN = sys.argv[1] if len(sys.argv) > 1 else "results/encoder"
OUT = sys.argv[2] if len(sys.argv) > 2 else "results/tables"
os.makedirs(OUT, exist_ok=True)
rng = np.random.default_rng(0)


def load():
    """Load all runs; if a (model, train_cond, seed) appears twice keep the LAST one."""
    seen = {}
    for p in glob.glob(os.path.join(IN, "*", "runs.jsonl")):
        for line in open(p, encoding="utf-8"):
            r = json.loads(line)
            seen[(r["model"], r["train_cond"], r["seed"])] = r
    return list(seen.values())


def holm(pvals):
    order = np.argsort(pvals)
    m = len(pvals)
    adj = np.empty(m)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * pvals[idx])
        adj[idx] = min(1.0, running)
    return adj


def fast_macro_f1(y, p, k):
    """Macro-F1 over k classes from integer arrays (sklearn convention: classes absent from
    both y and p contribute 0 only if present in the label set; here all k classes are used)."""
    tp = np.bincount(y[p == y], minlength=k).astype(float)
    pred = np.bincount(p, minlength=k).astype(float)
    true = np.bincount(y, minlength=k).astype(float)
    denom = pred + true
    # a class absent from BOTH y and p in a bootstrap resample is skipped, not counted as 0
    present = denom > 0
    f1 = 2 * tp[present] / denom[present]
    return f1.mean()


def boot_delta(preds_a, preds_b, labels, n_boot=10000):
    """Paired bootstrap over items: mean of Macro-F1(a) - Macro-F1(b) across seeds."""
    n = len(labels)
    labels = np.asarray(labels)
    k = int(labels.max()) + 1
    A = [np.asarray(p) for p in preds_a]
    B = [np.asarray(p) for p in preds_b]
    deltas = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        y = labels[idx]
        deltas[i] = np.mean([fast_macro_f1(y, a[idx], k) - fast_macro_f1(y, b[idx], k)
                             for a, b in zip(A, B)])
    return np.percentile(deltas, [2.5, 97.5])


def main():
    recs = load()
    if not recs:
        sys.exit("no runs found")
    rows = []
    for r in recs:
        for tc, m in r["tests"].items():
            rows.append({"model": r["model"], "train": r["train_cond"], "test": tc,
                         "seed": r["seed"], "acc": m["acc"], "macro_f1": m["macro_f1"],
                         "never_pred": m["n_classes_never_predicted"],
                         "n_truncated": m.get("n_truncated", -1),
                         "val_f1": r.get("val_macro_f1", np.nan), "collapsed": int(r.get("collapsed", False))})
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(OUT, "encoder_all_runs.csv"), index=False)

    agg = (df.groupby(["model", "train", "test"])
             .agg(n=("seed", "nunique"), acc_mean=("acc", "mean"), acc_std=("acc", "std"),
                  f1_mean=("macro_f1", "mean"), f1_std=("macro_f1", "std"),
                  never_pred=("never_pred", "max"),
                  n_truncated=("n_truncated", "max"),
                  val_f1_mean=("val_f1", "mean"), n_collapsed=("collapsed", "sum")).reset_index())
    agg.to_csv(os.path.join(OUT, "encoder_summary.csv"), index=False)

    # --- deltas
    piv = df.pivot_table(index=["model", "train", "seed"], columns="test", values="macro_f1").reset_index()
    out = []
    for name, (a, b) in {"D_script": ("cyrl", "latn"), "D_typo": ("latn_norm", "latn"),
                         "D_pure": ("cyrl", "latn_norm")}.items():
        for (model, train), g in piv.groupby(["model", "train"]):
            d = (g[a] - g[b]).values
            t, p = (stats.ttest_1samp(d, 0.0) if len(d) > 1 else (np.nan, np.nan))
            # bootstrap
            sub = [r for r in recs if r["model"] == model and r["train_cond"] == train]
            labels = sub[0]["tests"]["latn"]["labels"]
            ci = boot_delta([r["tests"][a]["preds"] for r in sub],
                            [r["tests"][b]["preds"] for r in sub], labels)
            out.append({"delta": name, "model": model, "train": train, "n_seeds": len(d),
                        "mean": d.mean(), "std": d.std(ddof=1) if len(d) > 1 else np.nan,
                        "p_raw_ttest": p, "boot_ci_lo": ci[0], "boot_ci_hi": ci[1]})
    od = pd.DataFrame(out)
    od["p_holm"] = np.nan
    for (dname, train), g in od.groupby(["delta", "train"]):
        od.loc[g.index, "p_holm"] = holm(g["p_raw_ttest"].fillna(1.0).values)
    od.to_csv(os.path.join(OUT, "encoder_deltas.csv"), index=False)
    pd.set_option("display.width", 200)
    print(agg.round(3).to_string(index=False))
    print(od.round(4).to_string(index=False))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile exp_llm_belebele.py
"""
exp_llm_belebele.py -- script robustness of decoder LLMs on Belebele (uzn_Latn) with
Latin / Latin-normalised / Cyrillic versions of the *whole* item
(passage, question, four options).

Scoring
-------
No free-form generation: the prompt ends with "Answer:" and we compare the next-token
logits of the four option letters A/B/C/D (standard, deterministic, no parsing errors).
Optional k-shot examples are drawn from OTHER Belebele items with a fixed seed and are
rendered in the SAME script as the test item.

Why this design
---------------
* Deterministic -> differences between conditions are not sampling noise.
* The instruction text is English and identical in all conditions, so only the Uzbek
  content changes script.
* Letter-position bias is recorded (pred_letter_counts) because it can interact with script.

Usage (Kaggle, 2xT4: use 4-bit for 7-9B models)
-----------------------------------------------
python prepare_data.py --out data --belebele
python exp_llm_belebele.py --model <hf-model-id> --shots 0 --load_4bit
python exp_llm_belebele.py --model <hf-model-id> --shots 5 --load_4bit

Model ids are an experimental choice -> [INFORMATION REQUIRED]. Decide the list before running
and keep it fixed (open-weight instruct models of 7-9B + at least one multilingual-strong model).
"""
import argparse, json, os, random, time
from collections import Counter

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

CONDS = ["latn", "latn_norm", "cyrl"]
LETTERS = ["A", "B", "C", "D"]


def load_cond(data_dir, cond):
    return [json.loads(l) for l in open(os.path.join(data_dir, f"belebele_{cond}.jsonl"), encoding="utf-8")]


def render(ex, with_answer):
    s = (f"P: {ex['flores_passage']}\nQ: {ex['question']}\n"
         f"A: {ex['mc_answer1']}\nB: {ex['mc_answer2']}\nC: {ex['mc_answer3']}\nD: {ex['mc_answer4']}\n"
         "Please choose the correct answer from the options above.\nAnswer:")
    if with_answer:
        s += " " + LETTERS[int(ex["correct_answer_num"]) - 1]
    return s


def letter_token_ids(tok):
    ids = []
    for L in LETTERS:
        cand = set()
        for form in (" " + L, L):
            t = tok.encode(form, add_special_tokens=False)
            if t:
                cand.add(t[-1])
        ids.append(sorted(cand))
    return ids


@torch.inference_mode()
def score_item(model, tok, prompt, ids_per_letter, max_ctx):
    """Returns (letter log-probs, truncated_flag, n_tokens).
    If the prompt is longer than max_ctx the START is cut (demonstrations first), keeping BOS and the
    END (question + options + 'Answer:'). Cyrillic needs more tokens than Latin, so truncation can
    differ between scripts -> it is recorded per item and MUST be reported."""
    enc = tok(prompt, return_tensors="pt", truncation=False)
    n_tok = int(enc["input_ids"].shape[1])
    truncated = n_tok > max_ctx
    if truncated:
        keep_bos = tok.bos_token_id is not None and int(enc["input_ids"][0, 0]) == tok.bos_token_id
        tail = max_ctx - 1 if keep_bos else max_ctx
        ids = enc["input_ids"][:, -tail:]
        am = enc["attention_mask"][:, -tail:]
        if keep_bos:
            ids = torch.cat([enc["input_ids"][:, :1], ids], 1)
            am = torch.cat([enc["attention_mask"][:, :1], am], 1)
        enc = {"input_ids": ids, "attention_mask": am}
    enc = {k: v.to(model.device) for k, v in enc.items() if k in ("input_ids", "attention_mask")}
    logits = model(**enc).logits[0, -1].float()
    logp = torch.log_softmax(logits, -1)
    return [torch.logsumexp(logp[ids], 0).item() for ids in ids_per_letter], truncated, n_tok


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--data_dir", default="data")
    ap.add_argument("--out_dir", default="results/llm")
    ap.add_argument("--shots", type=int, default=0)
    ap.add_argument("--seed", type=int, default=13)
    ap.add_argument("--max_ctx", type=int, default=6000)
    ap.add_argument("--limit", type=int, default=None, help="debug: first N items")
    ap.add_argument("--load_4bit", action="store_true")
    args = ap.parse_args()

    tok = AutoTokenizer.from_pretrained(args.model)
    kw = dict(torch_dtype=torch.float16, device_map="auto")
    if args.load_4bit:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    model = AutoModelForCausalLM.from_pretrained(args.model, **kw).eval()
    ids = letter_token_ids(tok)
    assert all(len(x) > 0 for x in ids), "could not find single tokens for answer letters"

    data = {c: load_cond(args.data_dir, c) for c in CONDS}
    n = len(data["latn"]) if args.limit is None else args.limit

    # ---- tokenizer sanity check: does the tokenizer silently drop characters of some script?
    # (a model that never SEES the Uzbek text answers by letter bias, not by reading; that is a
    #  tokenizer limitation, not "script robustness", and must be reported separately)
    tokcheck = {}
    for cond in CONDS:
        texts = [ex[k] for ex in data[cond][:100]
                 for k in ("flores_passage", "question", "mc_answer1", "mc_answer2", "mc_answer3", "mc_answer4")]
        kept = total = 0
        for t in texts:
            tids = tok(t, add_special_tokens=False)["input_ids"]
            a = "".join(t.split()).lower()
            b = "".join(tok.decode(tids, skip_special_tokens=True).split()).lower()
            total += len(a); kept += min(len(a), len(b))
        tokcheck[cond] = kept / max(total, 1)
    print("tokenizer character retention:", {k: round(v, 3) for k, v in tokcheck.items()}, flush=True)
    for cond, v in tokcheck.items():
        if v < 0.9:
            print(f"!!! WARNING: tokenizer of {args.model} drops {100*(1-v):.0f}% of characters in '{cond}'. "
                  f"Accuracy in this condition does NOT measure reading ability.", flush=True)
    rng = random.Random(args.seed)
    # few-shot indices fixed per item and shared across conditions (same demos in every script)
    shots = {i: rng.sample([j for j in range(len(data["latn"])) if j != i], args.shots)
             for i in range(n)} if args.shots else {i: [] for i in range(n)}

    os.makedirs(args.out_dir, exist_ok=True)
    tag = f"{args.model.replace('/', '__')}__{args.shots}shot" + ("__4bit" if args.load_4bit else "")
    out_path = os.path.join(args.out_dir, tag + ".jsonl")
    json.dump({"model": args.model, "shots": args.shots, "char_retention": tokcheck},
              open(os.path.join(args.out_dir, tag + ".tokcheck.json"), "w"))
    with open(out_path, "w", encoding="utf-8") as f:
        for cond in CONDS:
            correct, letters, t0, n_trunc = [], Counter(), time.time(), 0
            for i in range(n):
                demos = "\n\n".join(render(data[cond][j], True) for j in shots[i])
                prompt = (demos + "\n\n" if demos else "") + render(data[cond][i], False)
                lp, trunc, n_tok = score_item(model, tok, prompt, ids, args.max_ctx)
                pred = int(np.argmax(lp)) + 1
                gold = int(data[cond][i]["correct_answer_num"])
                correct.append(int(pred == gold)); letters[LETTERS[pred - 1]] += 1
                n_trunc += int(trunc)
                f.write(json.dumps({"model": args.model, "shots": args.shots, "cond": cond, "i": i,
                                    "pred": pred, "gold": gold, "logp": lp,
                                    "truncated": trunc, "n_tokens": n_tok}) + "\n")
            acc = float(np.mean(correct))
            print(f"[{tag}] {cond}: acc={acc:.4f} n={n} truncated={n_trunc} "
                  f"letters={dict(letters)} ({time.time()-t0:.0f}s)", flush=True)
            f.flush()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile analyze_llm.py
"""
analyze_llm.py -- accuracy per condition, Delta, paired bootstrap CI and exact McNemar test
(items are paired: same 900 questions in every script condition).

python analyze_llm.py results/llm results/tables
"""
import glob, json, os, sys
import numpy as np
import pandas as pd
from scipy.stats import binomtest

IN = sys.argv[1] if len(sys.argv) > 1 else "results/llm"
OUT = sys.argv[2] if len(sys.argv) > 2 else "results/tables"
os.makedirs(OUT, exist_ok=True)
rng = np.random.default_rng(0)


def holm(p):
    p = np.asarray(p); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for r, i in enumerate(o):
        run = max(run, (m - r) * p[i]); adj[i] = min(1, run)
    return adj


rows = []
for path in sorted(glob.glob(os.path.join(IN, "*.jsonl"))):
    recs = [json.loads(l) for l in open(path, encoding="utf-8")]
    df = pd.DataFrame(recs)
    model, shots = df["model"].iloc[0], int(df["shots"].iloc[0])
    corr = {c: (g.sort_values("i")["pred"].values == g.sort_values("i")["gold"].values).astype(int)
            for c, g in df.groupby("cond")}
    for a, b, name in [("cyrl", "latn", "D_script"), ("latn_norm", "latn", "D_typo"),
                       ("cyrl", "latn_norm", "D_pure")]:
        if a not in corr or b not in corr:
            continue
        x, y = corr[a], corr[b]
        d = x.mean() - y.mean()
        idx = rng.integers(0, len(x), (10000, len(x)))
        boots = x[idx].mean(1) - y[idx].mean(1)
        n01 = int(((x == 1) & (y == 0)).sum()); n10 = int(((x == 0) & (y == 1)).sum())
        p = binomtest(n01, n01 + n10, 0.5).pvalue if (n01 + n10) > 0 else 1.0
        tr = df.groupby("cond")["truncated"].sum().to_dict() if "truncated" in df else {}
        rows.append({"model": model, "shots": shots, "delta": name, "acc_a": x.mean(), "acc_b": y.mean(),
                     "truncated_a": tr.get(a, -1), "truncated_b": tr.get(b, -1),
                     "diff": d, "ci_lo": np.percentile(boots, 2.5), "ci_hi": np.percentile(boots, 97.5),
                     "mcnemar_p": p, "only_a_correct": n01, "only_b_correct": n10, "n": len(x)})
res = pd.DataFrame(rows)
if len(res):
    res["p_holm"] = np.nan
    for dname, g in res.groupby("delta"):
        res.loc[g.index, "p_holm"] = holm(g["mcnemar_p"].values)
res.to_csv(os.path.join(OUT, "llm_belebele_deltas.csv"), index=False)
pd.set_option("display.width", 220)
print(res.round(4).to_string(index=False))

In [ ]:
%%writefile exp_diagnostics.py
"""
exp_diagnostics.py -- WHY does the script change hurt?  Tokenizer + (optional) LM-likelihood diagnostics.

For every tokenizer/model and every condition (latn, latn_norm, cyrl) on SIB-200 sentences
(SIB-200 sentences are FLORES-200 sentences) and, optionally, an extra sentence file:

  tokens_per_word   fertility
  tokens_per_char   tokens per non-space character
  unk_rate          share of tokens equal to unk_token (if the tokenizer has one)
  bits_per_word     (optional) bits per WORD of a causal LM -> comparable across tokenisations and scripts

python exp_diagnostics.py --models xlm-roberta-base bert-base-multilingual-cased --data_dir data
python exp_diagnostics.py --models <causal-lm-id> --lm --data_dir data
python exp_diagnostics.py --models ... --extra_file flores_uzn_sentences.txt     # one sentence per line

FLORES note: FLORES-200 provides uzn_Latn; a Cyrillic Uzbek variant of FLORES was NOT found during
the literature/data check (only Southern Uzbek in Arabic script exists in FLORES+ dev)
[VERIFICATION REQUIRED before submission]. Therefore the Cyrillic text is synthetic.
"""
import argparse, glob, math, os
import numpy as np
import pandas as pd


def read_sib(data_dir, cond):
    texts = []
    for split in ("train", "validation", "test"):
        df = pd.read_csv(os.path.join(data_dir, f"sib200_{split}_{cond}.tsv"), sep="\t", quoting=3)
        texts += df["text"].astype(str).tolist()
    return texts


def char_retention(tok, texts, n=300):
    """Share of non-space characters that survive encode -> decode (special tokens such as <unk>
    are dropped by the decoder). 1.0 = lossless; a value near 0 means the tokenizer silently
    throws the text away (observed for uzlm/alloma-8B-Instruct on Cyrillic in the smoke run).
    Whitespace and case are ignored on purpose, so it measures lost CHARACTERS only."""
    kept = total = 0
    for t in texts[:n]:
        ids = tok(t, add_special_tokens=False)["input_ids"]
        dec = tok.decode(ids, skip_special_tokens=True)
        a = "".join(t.split()).lower()
        b = "".join(dec.split()).lower()
        total += len(a)
        kept += min(len(b), len(a))
    return kept / max(total, 1)


def tok_stats(tok, texts, max_lens=(128, 256)):
    n_tok = n_word = n_char = n_unk = 0
    over = {m: 0 for m in max_lens}
    for t in texts:
        ids = tok(t, add_special_tokens=False)["input_ids"]
        n_tok += len(ids)
        n_word += len(t.split())
        n_char += len(t.replace(" ", ""))
        for m in max_lens:  # +2 for [CLS]/[SEP]-like special tokens
            over[m] += int(len(ids) + 2 > m)
        if tok.unk_token_id is not None:
            n_unk += sum(1 for i in ids if i == tok.unk_token_id)
    r = {"tokens_per_word": n_tok / n_word,
         # per-character values are NOT strictly comparable across scripts: sh/ch/o'/g' are 2 Latin
         # characters but 1 Cyrillic letter. Words are identical across scripts -> use per word.
         "tokens_per_char": n_tok / n_char,
         "unk_rate": n_unk / max(n_tok, 1), "n_sent": len(texts), "n_words": n_word,
         "char_retention": char_retention(tok, texts)}
    for m in max_lens:
        r[f"share_truncated_at_{m}"] = over[m] / len(texts)
    return r


def bits_per_word(model_id, texts, limit=None):
    """Total negative log2-likelihood divided by the number of WORDS (identical across scripts),
    which is comparable across tokenisations and scripts; bits per character is not (see tok_stats)."""
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto").eval()
    nll = words = 0.0
    bos = tok.bos_token_id if tok.bos_token_id is not None else tok.eos_token_id
    for t in (texts if limit is None else texts[:limit]):
        ids = tok(t, add_special_tokens=False, return_tensors="pt")["input_ids"].to(model.device)
        ids = torch.cat([torch.tensor([[bos]], device=ids.device), ids], 1)
        with torch.inference_mode():
            logits = model(ids).logits[:, :-1].float()
        lp = torch.log_softmax(logits, -1).gather(-1, ids[:, 1:, None]).squeeze(-1)
        nll += -lp.sum().item()
        words += len(t.split())
    return nll / math.log(2) / words


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", nargs="+", required=True)
    ap.add_argument("--data_dir", default="data")
    ap.add_argument("--lm", action="store_true", help="also compute bits per word (causal LMs; run causal LMs in a separate call from encoders)")
    ap.add_argument("--extra_file", default=None)
    ap.add_argument("--out", default="results/tables/diagnostics.csv")
    a = ap.parse_args()
    from transformers import AutoTokenizer
    from translit_uz import lat2cyr, latin_normalised

    sets = {c: read_sib(a.data_dir, c) for c in ("latn", "latn_norm", "cyrl")}
    if a.extra_file:
        extra = [l.strip() for l in open(a.extra_file, encoding="utf-8") if l.strip()]
        sets.update({"extra_latn": extra, "extra_latn_norm": [latin_normalised(x) for x in extra],
                     "extra_cyrl": [lat2cyr(x) for x in extra]})
    rows = []
    for m in a.models:
        tok = AutoTokenizer.from_pretrained(m)
        for c, texts in sets.items():
            r = {"model": m, "cond": c, **tok_stats(tok, texts)}
            if a.lm:
                r["bits_per_word"] = bits_per_word(m, texts)
            rows.append(r)
            print(r, flush=True)
    os.makedirs(os.path.dirname(a.out), exist_ok=True)
    pd.DataFrame(rows).to_csv(a.out, index=False)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile analyze_deep.py
"""
analyze_deep.py -- extra analyses (printed to the log; also CSV):
  1. per-class Macro-F1 by (model, train_cond, test_cond), mean over seeds
  2. fertility (tokens/word) ratio cyrl/latn vs D_script (per model x train_cond), Spearman rho (n is tiny: descriptive only)
  3. error breakdown: share of items that flip correct->wrong / wrong->correct between latn and cyrl test (per model x train_cond)
usage: python analyze_deep.py results/encoder results/tables
"""
import glob, json, os, sys
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import f1_score

IN = sys.argv[1] if len(sys.argv) > 1 else "results/encoder"
OUT = sys.argv[2] if len(sys.argv) > 2 else "results/tables"
os.makedirs(OUT, exist_ok=True)

seen = {}
for p in glob.glob(os.path.join(IN, "*", "runs.jsonl")):
    for line in open(p, encoding="utf-8"):
        r = json.loads(line); seen[(r["model"], r["train_cond"], r["seed"])] = r
runs = list(seen.values())
if not runs:
    sys.exit("no runs")

pc, flip, dd = [], [], []
for r in runs:
    names = r["label_names"]; k = len(names)
    for tc, t in r["tests"].items():
        f = f1_score(t["labels"], t["preds"], average=None, labels=list(range(k)), zero_division=0)
        for i, n in enumerate(names):
            pc.append({"model": r["model"], "train": r["train_cond"], "test": tc, "seed": r["seed"], "class": n, "f1": f[i]})
    a = np.array(r["tests"]["latn"]["preds"]) == np.array(r["tests"]["latn"]["labels"])
    b = np.array(r["tests"]["cyrl"]["preds"]) == np.array(r["tests"]["cyrl"]["labels"])
    flip.append({"model": r["model"], "train": r["train_cond"], "seed": r["seed"],
                 "both_correct": (a & b).mean(), "lost_in_cyrl": (a & ~b).mean(),
                 "gained_in_cyrl": (~a & b).mean(), "both_wrong": (~a & ~b).mean(),
                 "d_script": r["tests"]["cyrl"]["macro_f1"] - r["tests"]["latn"]["macro_f1"]})

pc = pd.DataFrame(pc)
t1 = pc.groupby(["model", "train", "test", "class"]).f1.mean().unstack("test").round(3)
t1["cyrl-latn"] = (t1["cyrl"] - t1["latn"]).round(3)
t1.reset_index().to_csv(f"{OUT}/deep_per_class.csv", index=False)
fl = pd.DataFrame(flip)
t3 = fl.groupby(["model", "train"]).mean(numeric_only=True).drop(columns="seed").round(3)
t3.reset_index().to_csv(f"{OUT}/deep_error_flips.csv", index=False)

print("\n== per-class F1 (mean over seeds)"); print(t1.to_string())
print("\n== item-level flips latn->cyrl test (mean over seeds)"); print(t3.to_string())

diag = os.path.join(OUT, "diagnostics_encoders.csv")
if os.path.exists(diag):
    d = pd.read_csv(diag)
    w = d.pivot(index="model", columns="cond", values="tokens_per_word")
    if {"latn", "cyrl"} <= set(w.columns):
        w["fertility_ratio_cyrl/latn"] = w["cyrl"] / w["latn"]
        m = fl.groupby(["model", "train"]).d_script.mean().reset_index().merge(
            w[["fertility_ratio_cyrl/latn"]].reset_index(), on="model")
        m.round(4).to_csv(f"{OUT}/deep_fertility_vs_delta.csv", index=False)
        print("\n== fertility ratio vs D_script"); print(m.round(4).to_string(index=False))
        for tr, g in m.groupby("train"):
            if len(g) >= 3:
                rho, p = stats.spearmanr(g["fertility_ratio_cyrl/latn"], g["d_script"])
                print(f"train={tr}: Spearman rho={rho:.2f} p={p:.2f} (n={len(g)} models; descriptive only)")

In [ ]:
%%writefile check_apostrophes.py
"""
check_apostrophes.py -- counts apostrophe-like code points in the Latin Uzbek texts
(SIB-200 uzn_Latn train/validation/test and Belebele uzn_Latn: passage, question, answers).
Purpose: show whether the latn_norm (apostrophe-normalised) control can differ from latn at all.
usage: python check_apostrophes.py data
"""
import sys, glob, csv, unicodedata
from collections import Counter
D = sys.argv[1] if len(sys.argv) > 1 else "data"
APOS = ["'", "ʻ", "ʼ", "‘", "’", "`", "´", "ʹ", "′"]

def report(name, texts):
    c = Counter(); sent = Counter(); n = 0
    for t in texts:
        n += 1
        for ch in APOS:
            k = t.count(ch)
            if k: c[ch] += k; sent[ch] += 1
    print(f"\n[{name}] texts={n}")
    for ch in APOS:
        print(f"  U+{ord(ch):04X} {unicodedata.name(ch):<32} occurrences={c[ch]:6d}  texts_with={sent[ch]:5d}")
    return c

sib = []
for p in sorted(glob.glob(f"{D}/sib200_*_latn.tsv")):
    with open(p, encoding="utf-8") as f:
        r = csv.DictReader(f, delimiter="\t")
        sib += [row["text"] for row in r]
report("SIB-200 uzn_Latn (as used: latn)", sib)

try:
    from datasets import load_dataset
    ds = load_dataset("facebook/belebele", "uzn_Latn", split="test")
    fields = {"passage": "flores_passage", "question": "question",
              "answers": None}
    report("Belebele passage", [r["flores_passage"] for r in ds])
    report("Belebele question", [r["question"] for r in ds])
    report("Belebele answers", [r[f"mc_answer{i}"] for r in ds for i in range(1, 5)])
except Exception as e:
    print("Belebele check failed:", repr(e))

In [ ]:
run('python -m py_compile ' + ' '.join(f for f in os.listdir(WORK) if f.endswith('.py'))); print('skriptlar kompilyatsiya qilindi')

## O'girgich testi (29 ta test o'tishi kerak)

In [ ]:
run("python -m pytest -q test_translit.py")

## Ma'lumot tayyorlash (lotin / lotin-normallashgan / kirill)

In [ ]:
run("python prepare_data.py --out data" + (" --belebele" if RUN_LLM and LLM_MODELS else ""))
run("python check_apostrophes.py data")   # SIB-200 and Belebele: apostrophe code points

## Kodlovchilar (SIB-200): pilot, so'ng yakuniy; to'xtasa shu katakchani qayta ishga tushiring (resume)

In [ ]:
if RUN_ENCODERS:
    for m in ENCODER_MODELS:
        t0 = time.time()
        run(f"python exp_encoder_sib200.py --model {m} --data_dir data --out_dir results/encoder "
            f"--stage all --epochs {EPOCHS} --seeds {' '.join(map(str, SEEDS))} --lrs {' '.join(LRS)}")
        print(f"### {m}: {(time.time()-t0)/60:.1f} min", flush=True)
    run("python analyze_encoder.py results/encoder results/tables")
    run("python analyze_deep.py results/encoder results/tables")

## Tokenizator tashxisi

In [ ]:
if RUN_DIAGNOSTICS:
    run("python exp_diagnostics.py --models " + " ".join(ENCODER_MODELS) + " --data_dir data --out results/tables/diagnostics_encoders.csv")

## LLM (Belebele, 0-shot, 4-bit)

In [ ]:
if RUN_LLM and LLM_MODELS:
    for m in LLM_MODELS:
        for k in LLM_SHOTS:
            t0 = time.time()
            cmd = f"python exp_llm_belebele.py --model {m} --data_dir data --out_dir results/llm --shots {k}"
            if LLM_4BIT: cmd += " --load_4bit"
            if LLM_LIMIT: cmd += f" --limit {LLM_LIMIT}"
            run(cmd)
            print(f"### {m} {k}-shot: {(time.time()-t0)/60:.1f} min", flush=True)
    run("python analyze_llm.py results/llm results/tables")

## Natijalar

In [ ]:
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 50)
for f in sorted(glob.glob("results/tables/*.csv")):
    if "all_runs" in f: continue
    print("\n==", f); print(pd.read_csv(f).round(4).to_string(index=False)[:8000])
shutil.make_archive(os.path.join(WORK, "results"), "zip", WORK, "results")
print("\nTAYYOR:", os.path.join(WORK, "results.zip"))